### 라이브러리 선언하기

In [1]:
# 데이터 처리
import pandas as pd
import numpy as np

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

### 데이터 불러오기

In [22]:
### 데이터 정의
dataUrl = "https://raw.githubusercontent.com/hyokwan/python-lecture/refs/heads/master/dataset/cars.csv"
dataUrl = "https://raw.githubusercontent.com/hyokwan/python-lecture/refs/heads/master/dataset/customer.csv"
### 데이터 불러오기
featuresData = pd.read_csv(dataUrl)
featuresData.shape

(20000, 3)

### A. 전처리

### A-1. 데이터타입 통합 및 특성숫자변경

In [23]:
### TODO : 좌표계를 사용하는 모델인 경우 스케일러 필요 (MinMaxScaler)

In [42]:
# ★ 수정포인트 라벨 숫자 매핑
labelDict = { "normal":0, "diamond":1, "vip":2 }
featuresData["label_le"] = featuresData.label.map( labelDict )
featuresData

,balance,stock,label,label_le
0,30000000,22500000,normal,0
1,280000000,48000000,diamond,1
2,300000000,40666666,diamond,1
3,54000000,28000000,normal,0
4,768000000,32000000,vip,2
...,...,...,...,...
19995,628000000,44666666,diamond,1
19996,276000000,20000000,normal,0
19997,652000000,41333333,diamond,1
19998,676000000,45333333,diamond,1


### A-2. 특성선정 및 데이터 분리

In [32]:
# 상관분석 corr r값 
corrDf = featuresData.corr(numeric_only=True)
corrDf

,balance,stock,label_le
balance,1.000000,0.565942,0.883144
stock,0.565942,1.000000,0.824174
label_le,0.883144,0.824174,1.000000


In [45]:
# ★수정포인트 정답지 컬럼
label = ["label_le"]
corrDf = featuresData.corr(numeric_only=True)
stdCorr = 0.5
features = list( corrDf.loc [ ( abs( corrDf[label[0]])  >= stdCorr) &  ( corrDf[label[0]]  != 1) ].index )

print(f"문제지: {features} 정답지: {label} ")

문제지: ['balance', 'stock'] 정답지: ['label_le'] 


In [73]:
### 데이터 분리
trainData, testData = \
        train_test_split( featuresData, test_size=0.2, random_state=10)

In [74]:
trainDataFeatures = trainData.loc[:, features]
trainDataLabel = trainData.loc[:, ["label"]]
testDataFeatures = testData.loc[:, features]
testDataLabel = testData.loc[:, ["label"]]
print( trainDataFeatures.shape )
print( trainDataLabel.shape )
print( testDataFeatures.shape )
print( testDataLabel.shape )

(16000, 2)
(16000, 1)
(4000, 2)
(4000, 1)


### B. 모델 학습

### B-1. 모델 정의 및 학습

In [75]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
## 모델정의
knnModel = KNeighborsClassifier(n_neighbors=5)
dtModel = DecisionTreeClassifier(random_state=5)
## 모델학습
fittedKnnModel = knnModel.fit( trainDataFeatures, trainDataLabel  )
fittedDtModel = dtModel.fit(trainDataFeatures, trainDataLabel  )

c:\Users\hk\fintech_data_2026\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


### C. 예측

### C-1. 예측

In [80]:
# diamond 280000000	48000000

inBalance = 280000000
inStock = 48000000
testDf = pd.DataFrame([[  inBalance,inStock   ]])
fittedDtModel.predict( testDf)

c:\Users\hk\fintech_data_2026\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


array(['diamond'], dtype=object)

### C-2. 데이터 정리